# Suite TS — The TypeScript language

TypeScript and JavaScript as one tree language (`mbse.Programs.TypeScript.Syntax`), whose kinds follow
typescript-estree: its kinds and categories, the identifiers it accepts, each kind's own checks, and where each kind
and feature exists, in ECMAScript's editions and TypeScript's versions, which the standards check.

In [ ]:
from mbse.Programs.TypeScript import ECMAScript2020, ECMAScript2025, Syntax as S, TypeScript50, TypeScript59
from mbse.Programs.TypeScript._Standard import ECMAScriptStandard, TypeScriptStandard

L = S.LANGUAGE


def n(name):
    return S.Identifier(name=name)


def lit(raw):
    return S.Literal(raw=raw)


def annotated(name, type_):
    return S.Identifier(name=name, typeAnnotation=S.TSTypeAnnotation(typeAnnotation=type_))

## TS-01 · Every kind belongs to the language, the grammar lists them, and their fields are in source order

In [ ]:
kinds = L.kinds()
assert list(kinds) == [k.__name__ for k in S.KINDS] and len(kinds) == 163
assert list(L.categories()) == ["Statement", "Expression", "Pattern", "ClassElement", "TypeElement", "TypeNode"]
assert all(k.NAME == f"Programs.TypeScript.{k.KIND}" for k in S.KINDS)
grammar = L.grammar()
assert grammar["language"] == "TypeScript" and grammar["base"] == {"ECMAScript": 2015, "TypeScript": 100}
satisfies = next(k for k in grammar["kinds"] if k["kind"] == "TSSatisfiesExpression")
assert satisfies["since"] == {"TypeScript": 409} and satisfies["category"] == "Expression"
assert next(k for k in grammar["kinds"] if k["kind"] == "JSXElement")["extension"] is True
assert [k.__name__ for k in S.KINDS if k.EXTENSION] == [k.__name__ for k in S.KINDS if k.__name__.startswith("JSX")]
assert S.es(2020, 307) == {"ECMAScript": 2020, "TypeScript": 307} and S.ts(509) == {"TypeScript": 509}
assert isinstance(S.Comment(), S.Statement) and isinstance(S.TSAnyKeyword(), S.TypeNode)
assert [f.name for f in S.ClassDeclaration.FIELDS] == [
    "decorators", "declare", "abstract", "id", "typeParameters", "superClass", "superTypeArguments", "implements",
    "body"]
assert [f.name for f in S.MethodDefinition.FIELDS] == [
    "decorators", "accessibility", "static", "override", "methodKind", "computed", "key", "optional", "value"]
assert [f.name for f in S.Identifier.FIELDS] == ["decorators", "name", "optional", "typeAnnotation"]
print(len(kinds), "kinds")

## TS-02 · Identifiers are Unicode identifiers, `$` included; keywords are names to the tree

In [ ]:
for name in ("x_1", "$", "$el", "_", "Größe", "async", "of", "class", "this"):
    assert L.validate(n(name)) == [], name
    assert L.validate(S.PrivateIdentifier(name=name)) == [], name
assert L.validate(n("1x")) == ["'1x' is not an identifier"]
assert L.validate(n("a-b")) == ["'a-b' is not an identifier"]
assert L.validate(n("")) == ["'' is not an identifier"]
assert L.validate(S.PrivateIdentifier(name="#x")) == ["'#x' is not an identifier"]
assert L.validate(n(3)) == ["Identifier.name must be a str, got int"]
assert L.validate(S.PrivateIdentifier(name=3)) == ["PrivateIdentifier.name must be a str, got int"]
assert L.validate(S.Identifier()) == ["an Identifier needs a name"]
print("ok")

## TS-03 · Each kind's own checks: what its lists must hold, and what goes together

In [ ]:
loop = S.WhileStatement(test=n("a"), body=S.EmptyStatement())
problems = {
    lit(""): "a Literal needs a raw",
    S.TemplateLiteral(): "a TemplateLiteral needs one more quasi than expressions",
    S.TemplateLiteral(quasis=[S.TemplateElement(raw="a")], expressions=[n("b")]):
        "a TemplateLiteral needs one more quasi than expressions",
    S.SequenceExpression(expressions=[n("a")]): "a SequenceExpression needs two expressions",
    S.TryStatement(block=S.BlockStatement()): "a TryStatement needs a handler or a finalizer",
    S.VariableDeclaration(declarationKind="let"): "a VariableDeclaration needs a declarator",
    S.VariableDeclarator(id=n("a"), definite=True): "definite needs a name with a type annotation",
    S.TSTypeParameterDeclaration(): "a TSTypeParameterDeclaration needs a type parameter",
    S.TSTemplateLiteralType(): "a TSTemplateLiteralType needs one more quasi than types",
    S.TSUnionType(types=[S.TSAnyKeyword()]): "a TSUnionType needs two types",
    S.TSIntersectionType(types=[S.TSAnyKeyword()]): "a TSIntersectionType needs two types",
}
for node, problem in problems.items():
    assert L.validate(node) == [problem], (node, L.validate(node))
# `if (a) if (b) x; else y;` would give the else to the inner `if`, through loops, labels and `with` too
dangling = "an if without else ends the consequent, and would take the else"
for consequent in (S.IfStatement(test=n("b"), consequent=S.EmptyStatement()),
                   S.LabeledStatement(label=n("l"), body=S.ForOfStatement(
                       left=n("x"), right=n("y"), body=S.IfStatement(test=n("b"), consequent=S.EmptyStatement()))),
                   S.IfStatement(test=n("b"), consequent=S.EmptyStatement(), alternate=S.WithStatement(
                       object=n("o"), body=S.IfStatement(test=n("c"), consequent=S.EmptyStatement())))):
    node = S.IfStatement(test=n("a"), consequent=consequent, alternate=S.EmptyStatement())
    assert L.validate(node) == [dangling], L.validate(node)
fine = [
    S.IfStatement(test=n("a"), consequent=loop, alternate=S.EmptyStatement()),
    S.IfStatement(test=n("a"), consequent=S.IfStatement(test=n("b"), consequent=S.EmptyStatement(),
                                                        alternate=S.EmptyStatement()), alternate=S.EmptyStatement()),
    S.IfStatement(test=n("a"), consequent=S.IfStatement(test=n("b"), consequent=S.EmptyStatement())),
    S.TemplateLiteral(quasis=[S.TemplateElement(raw="a"), S.TemplateElement(raw="b", tail=True)],
                      expressions=[n("b")]),
    S.TryStatement(block=S.BlockStatement(), finalizer=S.BlockStatement()),
    S.VariableDeclarator(id=annotated("a", S.TSNumberKeyword()), definite=True),
    S.TSUnionType(types=[S.TSAnyKeyword(), S.TSNullKeyword()]),
]
for node in fine:
    assert L.validate(node) == [], (node, L.validate(node))
assert L.validate(S.Literal(raw=1)) == ["Literal.raw must be a str, got int"]
assert L.validate(S.Literal()) == ["a Literal needs a raw"] and S.Literal(raw=1).features() == []
print(len(problems), "problems")

## TS-04 · Where kinds and features exist: since which edition of ECMAScript, and which version of TypeScript

In [ ]:
def needs(node, problem, fine, lacking):
    """`node` is fine in the standard `fine`, and has `problem` in the standard `lacking`."""
    assert fine.check(node) == [], (node, fine.check(node))
    assert lacking.check(node) == [problem], (lacking, lacking.check(node))


assert TypeScript59.STANDARD.name() == "TypeScript 5.9" and TypeScript59.JSX.name() == "TypeScript 5.9 with JSX"
assert ECMAScript2025.STANDARD.name() == "ES2025" and ECMAScript2020.JSX.name() == "ES2020 with JSX"
assert repr(TypeScript50.STANDARD) == "<standard TypeScript 5.0>"
assert TypeScript59.STANDARD.version == 509 and ECMAScript2025.STANDARD.version == 2025
ES2015, ES2016, ES2019, ES2021, ES2022 = (ECMAScriptStandard(y) for y in (2015, 2016, 2019, 2021, 2022))
ES2020, ES2025 = ECMAScript2020.STANDARD, ECMAScript2025.STANDARD
TS39, TS44, TS49, TS50, TS59 = (TypeScriptStandard(5 if m >= 50 else 4 if m >= 40 else 3, m % 10)
                                for m in (39, 44, 49, 50, 59))
TS59 = TypeScript59.STANDARD
needs(S.TSAnyKeyword(), "TSAnyKeyword is not ECMAScript", TS59, ES2025)
needs(S.TSSatisfiesExpression(expression=n("a"), typeAnnotation=S.TSAnyKeyword()),
      "TSSatisfiesExpression needs TypeScript 4.9", TS49, TypeScriptStandard(4, 8))
needs(S.PrivateIdentifier(name="x"), "PrivateIdentifier needs ES2022", ES2022, ES2021)
needs(S.ChainExpression(expression=n("a")), "ChainExpression needs ES2020", ES2020, ECMAScriptStandard(2019))
needs(S.BinaryExpression(left=n("a"), operator="**", right=n("b")), "BinaryExpression.operator '**' needs ES2016",
      ES2016, ES2015)
needs(S.AssignmentExpression(left=n("a"), operator="??=", right=n("b")),
      "AssignmentExpression.operator '??=' needs ES2021", ES2021, ES2020)
needs(S.LogicalExpression(left=n("a"), operator="??", right=n("b")), "LogicalExpression.operator '??' needs ES2020",
      ES2020, ECMAScriptStandard(2019))
needs(lit("1n"), "bigint Literal needs ES2020", ES2020, ECMAScriptStandard(2019))
needs(lit("1_000"), "Literal with numeric separators needs ES2021", ES2021, ES2020)
assert ES2015.check(lit("1.5")) == [] and ES2015.check(lit("'a_n'")) == [] and ES2015.check(lit("._5")) == [
    "Literal with numeric separators needs ES2021"]
needs(S.MetaProperty(meta=n("import"), property=n("meta")), "import.meta needs ES2020", ES2020,
      ECMAScriptStandard(2019))
assert ES2015.check(S.MetaProperty(meta=n("new"), property=n("target"))) == [] and ES2015.check(S.MetaProperty()) == []
needs(S.MemberExpression(object=n("a"), property=n("b"), optional=True), "MemberExpression with ?. needs ES2020",
      ES2020, ECMAScriptStandard(2019))
needs(S.CallExpression(callee=n("a"), optional=True), "CallExpression with ?. needs ES2020", ES2020,
      ECMAScriptStandard(2019))
private_in = S.BinaryExpression(left=S.PrivateIdentifier(name="x"), operator="in", right=n("o"))
assert ES2022.check(private_in) == []
assert ES2021.check(private_in) == ["BinaryExpression of #name in needs ES2022", "left: PrivateIdentifier needs ES2022"]
needs(S.CatchClause(body=S.BlockStatement()), "CatchClause without a param needs ES2019", ES2019,
      ECMAScriptStandard(2018))
needs(S.VariableDeclaration(declarationKind="using", declarations=[S.VariableDeclarator(id=n("a"))]),
      "VariableDeclaration.declarationKind 'using' is not ECMAScript", TS59, ES2025)
needs(S.ImportDeclaration(source=lit("'m'"), phase="defer"), "ImportDeclaration.phase 'defer' needs TypeScript 5.9",
      TS59, TS50)
needs(S.Identifier(name="a", optional=True), "Identifier.optional is not ECMAScript", TS59, ES2025)
needs(S.ArrayPattern(optional=True), "ArrayPattern.optional is not ECMAScript", TS59, ES2025)
needs(S.TSTypeParameter(name=n("T"), isIn=True), "TSTypeParameter.isIn needs TypeScript 4.7", TS49, TS44)
# JSX is an extension, outside every edition and version: a standard has it with `jsx`
assert ES2015.check(S.JSXText(raw="a")) == ["JSXText needs JSX"]
assert ES2015.problems(S.JSXText(raw="a")) == ["JSXText needs JSX"]
assert ECMAScriptStandard(2015, jsx=True).problems(S.JSXText(raw="a")) == []
cast = S.TSTypeAssertion(typeAnnotation=S.TSAnyKeyword(), expression=n("a"))
assert TypeScript59.JSX.problems(cast) == ["TSTypeAssertion is not allowed with JSX"] and TS59.problems(cast) == []
assert TypeScriptStandard(4, 8).check(S.Program(body=[S.ExpressionStatement(expression=S.TSSatisfiesExpression(
    expression=n("a"), typeAnnotation=S.TSAnyKeyword()))])) == [
    "body[0].expression: TSSatisfiesExpression needs TypeScript 4.9"]
print("ok")